In [ ]:
import subprocess, sys
for pkg in ['earthaccess', 'rioxarray', 'pystac-client']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

import numpy as np
import pandas as pd
import earthaccess
import rioxarray as rxr
from pathlib import Path
from datetime import date, timedelta
from pystac_client import Client

try:
    auth = earthaccess.login(strategy='netrc')
    print('Earthdata auth: OK')
except Exception:
    auth = earthaccess.login(strategy='interactive')

fs = earthaccess.get_fsspec_https_session()
print('Imports OK')

In [ ]:
STATE_BBOX = {
    'Iowa':      [-96.64, 40.37, -90.14, 43.50],
    'Colorado':  [-109.06, 36.99, -102.04, 41.00],
    'Wisconsin': [-92.89, 42.49, -86.25, 47.08],
    'Missouri':  [-95.77, 35.99, -89.10, 40.61],
    'Nebraska':  [-104.05, 39.99, -95.31, 43.00],
}

FORECAST_DATES = {
    'aug1':  '08-01',
    'sep1':  '09-01',
    'oct1':  '10-01',
    'final': '11-01',
}
WINDOW_DAYS = 8   # MOD13Q1 is 16-day composite; wider window improves hit rate

YEARS = range(2005, 2015)   # backfill only; HLS covers 2015+

STAC_URL   = 'https://cmr.earthdata.nasa.gov/stac/LPCLOUD'
COLLECTION = 'MOD13Q1.061'
NDVI_BAND  = '250m_16_days_NDVI'
SCALE      = 10000.0

OUT_DIR = Path('../data/raw')
OUT_DIR.mkdir(parents=True, exist_ok=True)
catalog = Client.open(STAC_URL, headers={})
print('Config loaded')

In [ ]:
def search_modis_tiles(bbox, date_str, year, window=WINDOW_DAYS):
    target = date(year, int(date_str[:2]), int(date_str[3:]))
    start  = (target - timedelta(days=window)).isoformat()
    end    = (target + timedelta(days=window)).isoformat()
    results = catalog.search(
        collections=[COLLECTION],
        bbox=bbox,
        datetime=f'{start}/{end}',
        max_items=10,
    )
    return list(results.items())

def compute_ndvi_modis(item, fs_session):
    try:
        url = item.assets[NDVI_BAND].href
        with fs_session.open(url) as f:
            da = rxr.open_rasterio(f, masked=True).squeeze()
        da = da.where(da != -3000).astype(float) / SCALE
        da = da.where((da >= -1) & (da <= 1))
        return float(da.mean().values)
    except Exception as e:
        print(f'    MODIS NDVI error: {e}')
        return np.nan

print('Helpers defined')

In [ ]:
records = []

for state, bbox in STATE_BBOX.items():
    print(f'\n\u2500\u2500 {state} \u2500\u2500')
    for year in YEARS:
        for label, date_str in FORECAST_DATES.items():
            items = search_modis_tiles(bbox, date_str, year)
            if not items:
                print(f'  {year} {label}: no tiles')
                records.append({'state': state, 'year': year,
                                'forecast_date': label, 'ndvi_mean': np.nan})
                continue
            vals = [compute_ndvi_modis(item, fs) for item in items[:4]]
            vals = [v for v in vals if not np.isnan(v)]
            ndvi = float(np.nanmedian(vals)) if vals else np.nan
            print(f'  {year} {label}: {len(vals)} tiles \u2192 NDVI={ndvi:.3f}')
            records.append({'state': state, 'year': year,
                            'forecast_date': label, 'ndvi_mean': ndvi})

print(f'\nDone. {len(records)} records.')

In [ ]:
long_df = pd.DataFrame(records)

wide_df = (
    long_df
    .pivot_table(index=['state','year'], columns='forecast_date', values='ndvi_mean')
    .reset_index()
)
wide_df.columns.name = None
wide_df = wide_df.rename(columns={
    'aug1': 'ndvi_aug1', 'sep1': 'ndvi_sep1',
    'oct1': 'ndvi_oct1', 'final': 'ndvi_final'
})
ndvi_cols = ['ndvi_aug1','ndvi_sep1','ndvi_oct1','ndvi_final']
wide_df[ndvi_cols] = wide_df[ndvi_cols].interpolate(axis=1, limit_direction='both')

wide_df.to_csv(OUT_DIR / 'ndvi_modis_2005_2014.csv', index=False)
print(f'Saved ndvi_modis_2005_2014.csv ({len(wide_df)} rows)')
print(wide_df.head())

In [ ]:
hls_path   = OUT_DIR / 'ndvi_by_state_date.csv'
modis_path = OUT_DIR / 'ndvi_modis_2005_2014.csv'

if not hls_path.exists():
    raise FileNotFoundError(
        "ndvi_by_state_date.csv not found. "
        "Commit the SageMaker output from 03_satellite.ipynb first."
    )

hls_df   = pd.read_csv(hls_path)
modis_df = pd.read_csv(modis_path)

combined = (
    pd.concat([modis_df, hls_df], ignore_index=True)
    .sort_values(['state','year'])
    .reset_index(drop=True)
)
combined.to_csv(OUT_DIR / 'ndvi_combined.csv', index=False)
print(f'Saved ndvi_combined.csv ({len(combined)} rows, years {combined.year.min()}\u2013{combined.year.max()})')
print(combined.groupby('state')['year'].agg(['min','max','count']))

**Note:** MODIS MOD13Q1 resolution is 250m vs. HLS 30m.
Both are valid vegetation indices for state-level modeling. The resolution difference is
immaterial when averaging across an entire state bounding box.